%md
# Delta Lake MERGE — Station Metadata SCD
### Station metadata (elevation, location) genuinely changes over time in production due to re-surveys, relocations, new stations coming online. Our source file is a static snapshot, so this notebook simulates a realistic update batch and uses Delta's `MERGE INTO` to demonstrate handling it correctly: updating changed stations and inserting new ones in a single atomic operation.

### Grabbing one station's current metadata so we have something concrete to compare against after the merge.

In [0]:
df_stations_current = spark.table("climate_project.silver_stations")
df_stations_current.filter(df_stations_current.ID == "AG000060390").show()

+-----------+--------+---------+---------+-----+------------------+
|         ID|LATITUDE|LONGITUDE|ELEVATION|STATE|              NAME|
+-----------+--------+---------+---------+-----+------------------+
|AG000060390| 36.7167|     3.25|     24.0|     |ALGER-DAR EL BEIDA|
+-----------+--------+---------+---------+-----+------------------+



### Pretend a refreshed station file just arrived: one existing station got re-surveyed (elevation changed) and one brand-new station was added to the network.

In [0]:
from pyspark.sql import Row

df_station_updates = spark.createDataFrame([
    Row(ID="AG000060390", LATITUDE=36.7167, LONGITUDE=3.25, ELEVATION=26.5, STATE=None, NAME="ALGER-DAR EL BEIDA"),
    Row(ID="TESTNEW00001", LATITUDE=40.0, LONGITUDE=-105.0, ELEVATION=1600.0, STATE="CO", NAME="TEST NEW STATION")
])

df_station_updates.show()

+------------+--------+---------+---------+-----+------------------+
|          ID|LATITUDE|LONGITUDE|ELEVATION|STATE|              NAME|
+------------+--------+---------+---------+-----+------------------+
| AG000060390| 36.7167|     3.25|     26.5| NULL|ALGER-DAR EL BEIDA|
|TESTNEW00001|    40.0|   -105.0|   1600.0|   CO|  TEST NEW STATION|
+------------+--------+---------+---------+-----+------------------+



### `MERGE INTO` updates matched rows and inserts unmatched ones (the new station) in a single operation. It is safer than writing separate update/insert logic by hand, and safe to re-run without creating duplicates.

In [0]:
spark.sql("""
MERGE INTO climate_project.silver_stations AS target
USING station_updates AS source
ON target.ID = source.ID
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""").show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|                2|               2|               0|                0|
+-----------------+----------------+----------------+-----------------+



### Confirms that the merge worked: `AG000060390`'s elevation updated to 26.5, and `TESTNEW00001` exists exactly once, confirming the merge is safe to re-run.

In [0]:
df_check = spark.table("climate_project.silver_stations")
df_check.filter(df_check.ID.isin("AG000060390", "TESTNEW00001")).show()

+------------+--------+---------+---------+-----+------------------+
|          ID|LATITUDE|LONGITUDE|ELEVATION|STATE|              NAME|
+------------+--------+---------+---------+-----+------------------+
|TESTNEW00001|    40.0|   -105.0|   1600.0|   CO|  TEST NEW STATION|
| AG000060390| 36.7167|     3.25|     26.5| NULL|ALGER-DAR EL BEIDA|
+------------+--------+---------+---------+-----+------------------+

